# MRL prediction for JURA sequence groups

Loads the same sequence groups as `JURA_analysis.ipynb` (`NT96`/`NT192`/`NT384`/`Human`/
`human_truncated`/`Random_gc50`/`Random_gc_adjusted`) and runs each group through the trained
ribosome-loading CNN (`models/optimus5prime.py`) to add a predicted `MRL` column to every
group's dataframe. Split out of `JURA_analysis.ipynb` to keep that notebook focused on the
sequence-derived markers, and this one focused on the model-based prediction.

## 0. Setup

In [1]:
import sys, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append("..")   # matches the convention used in model.ipynb / test.ipynb
sys.path.append(".")

import src.constants as constants
import src.rna_analysis as rna
import src.KLD_calculation as kldcalc
import src.rng_sequences as rngseq

sns.set_style("whitegrid")
plt.rc("font", size=12)

# Marker 3 (MFE) needs ViennaRNA - check it's importable up front so we can skip that section cleanly if not.
try:
    import ViennaRNA as vrna
    MFE_AVAILABLE = True
except ImportError:
    MFE_AVAILABLE = False
    warnings.warn("ViennaRNA is not installed - marker 3 (MFE) will be skipped. "
                   "Install it with: pip install ViennaRNA")
print("Setup complete. MFE/ViennaRNA available:", MFE_AVAILABLE)


Setup complete. MFE/ViennaRNA available: True


## 1. Configuration

Paths, column names, and the ID rule used to pull `human_truncated` out of the twist pool file.
Adjust these if your real column names differ.

In [2]:
# --- File locations -------------------------------------------------------
NEW_DATASET_PATH = "data/JURA data/new_dataset.csv"            # contains 'human' and 'random' groups
TWIST_POOL_PATH  = "data/JURA data/delivered_twist_pool.csv"   # contains 'human_truncated', hidden by ID
JURA_DATASET_PATH = "data/JURA data/samples_by_library.csv"    # contains 'NT96', 'NT192' and 'NT384' group

# --- Column names -----------------------------------------------------------
SEQ_COL   = "Sequence"   # sequence column name used everywhere downstream
GROUP_COL = "Group"      # group label column name used everywhere downstream
MIN_SEQ_LEN = 50 
TWIST_SEQ_COL    = "sequence"      # sequence column name *inside the twist pool file* - adjust if different there
TWIST_ID_COL     = "transcript_id" # identifier column inside the twist pool file
TWIST_ID_PREFIX  = "ENST"          # rows whose ID starts with this are human_truncated
TWIST_GROUP_NAME = "human_truncated"
PROBABILITIES = [0.201375,0.292513,0.303283,0.202829] # probabilities for ACGT in human_truncated
GROUPS       = ["NT96", "NT192", "NT384", "Human", "human_truncated", "Random_gc50","Random_gc_adjusted"]
PLOT_GROUPS = [g for g in GROUPS if g != "Human"]
START_CODONS = ["AUG", "CUG", "GUG", "UUG", "ACG"]   # used for the uORF marker

MFE_SAMPLE_SIZE = 150     # ViennaRNA folding is slow; subsample large groups for marker 3.
                          # Set to None to use every sequence in every group.
RANDOM_SEED = 42

PALETTE = dict(zip(GROUPS, sns.color_palette("Set2", n_colors=len(GROUPS))))

## 3. Load sequence data

### 3a. `new_dataset.csv` → `human` + `random`
Load step, then a separate filter step (kept apart so it's clear which rows came from where).

In [3]:
# Load: read the raw file (or fall back to a synthetic stand-in if it isn't present yet)
new_dataset = pd.read_csv(NEW_DATASET_PATH)
if "SeqID" in new_dataset.columns:
    new_dataset.set_index("SeqID", inplace=True)


print(f"Raw new_dataset: {len(new_dataset)} rows; groups present: {sorted(new_dataset[GROUP_COL].unique())}")


Raw new_dataset: 441887 rows; groups present: ['Human', 'Random_dl_gc50', 'Random_dl_gc_adjusted', 'Random_gc50', 'Random_gc_adjusted']


In [4]:
# Filter: new_dataset.csv already carries an explicit Group column - just keep 'human' and 'random'
# (it may contain other groups we don't want here, e.g. earlier random-control variants).
humandf = new_dataset.loc[new_dataset[GROUP_COL].isin(["Human"]), [SEQ_COL, GROUP_COL]].copy()
random50df = pd.DataFrame(rngseq.randomseqs_type2(10000,200,"Random_gc50",seed=RANDOM_SEED),columns=['Sequence','Group']) # 10000 sequences, length=200nt
human_random_df = pd.concat([humandf,random50df])
human_random_df

,Sequence,Group
ENST00000641515.2,CCCAGAUCUCUUCAGUUUUUAUGCCUCAUUCUGUGAAAAUUGCUGU...,Human
ENST00000616016.5,GGCGGCGGAGUCUCCCAAGUCCCCGCCGGGCGGGCGCGCGCCAGUG...,Human
ENST00000342066.8,GCAGAGCCCAGCAGAUCCCUGCGGCGUUCGCGAGGGUGGGACGGGA...,Human
ENST00000338591.8,GGGAGUGAGCGACACAGAGCGGGCCGCCACCGCCGAGCAGCCCUCC...,Human
ENST00000379410.8,AGGAGGCUGUGGACAGGGACCCAGACUUGCCGACCUGUACGACUCU...,Human
...,...,...
9995,UUGUUGCCUUCGUGUGGCCUCCCACGGUAAAACAUGUGCAAGCGUC...,Random_gc50
9996,ACGUCGAAAUGCCUCGGAUAUUAAUAGGUGUCCGUAGAUGAGACAU...,Random_gc50
9997,UCCACCCCGUCUUUAUGCAUUAGGCGUCAUAAUAUCUCAACGCAGG...,Random_gc50
9998,UACACUUUACCUGAUGUUUAACCGAAAUUUGCGCAUGCACGGCUUC...,Random_gc50


In [5]:
randomadjdf=random50df = pd.DataFrame(rngseq.randomseqs_type2(10000,200,"Random_gc_adjusted",probabilities=PROBABILITIES,seed=RANDOM_SEED),columns=['Sequence','Group']) # 10000 sequences, length=200nt
human_random_df = pd.concat([human_random_df,randomadjdf])

In [6]:
human_random_df.groupby(GROUP_COL).size()
print(f"new_dataset: {len(human_random_df)} rows; groups present: {sorted(human_random_df[GROUP_COL].unique())}")


new_dataset: 61887 rows; groups present: ['Human', 'Random_gc50', 'Random_gc_adjusted']


### 3b. `delivered_twist_pool.csv` → `human_truncated`
Load step, then the ID-based filter step described above.

In [7]:
# Load: read the raw twist pool (or fall back to a synthetic stand-in)
twist_pool = pd.read_csv(TWIST_POOL_PATH)
print(f"Raw twist pool: {len(twist_pool)} rows; columns: {list(twist_pool.columns)}")

Raw twist pool: 112985 rows; columns: ['Unnamed: 0', 'sequence', 'reads_norm', 'oligo_id', 'transcript_id']


In [8]:
# Filter: human_truncated is *hidden* in this pool - it isn't a labelled group, it's every row whose
# transcript_id is an Ensembl transcript ID (starts with 'ENST'). Every other ID prefix in this file
# belongs to a different design and is dropped here, in this dedicated filtering step.
is_human_truncated = twist_pool[TWIST_ID_COL].astype(str).str.startswith(TWIST_ID_PREFIX)

human_truncated_df = twist_pool.loc[is_human_truncated, [TWIST_SEQ_COL]].rename(columns={TWIST_SEQ_COL: SEQ_COL})
human_truncated_df[GROUP_COL] = TWIST_GROUP_NAME

print(f"{is_human_truncated.sum()} / {len(twist_pool)} rows started with '{TWIST_ID_PREFIX}' "
      f"and were kept as '{TWIST_GROUP_NAME}'")
human_truncated_df.head()


67935 / 112985 rows started with 'ENST' and were kept as 'human_truncated'


,Sequence,Group
0,CCTGTCTGAGCTGGAAACACAGCTTAGCTTCTAGACATCGCTGGCA...,human_truncated
2,GTCCTTGTGTCCGGCCTACCTGATAAGCCACTTGCCGACTGCTGTT...,human_truncated
6,AGAGTCCAGCCGGGGGTGCCGCACCCACTGGGGAGTGGGGAGGGAG...,human_truncated
7,AGTGCTTCTTAGGCCCACCCAAAAGGGCAATGGACCTCTCTGTATG...,human_truncated
11,GGACTGCTCACAGTCCCCCTTCGGAAAGACACTGCTACATGTTCTT...,human_truncated


### 3c. `NT96` / `NT192` / `NT384` 

In [9]:
jura_dataset=pd.read_csv(JURA_DATASET_PATH)
jura_dataset.rename(columns={'library':GROUP_COL,'sequence':SEQ_COL}, inplace=True)
jura_dataset.head()


,Sequence,Group
0,GGATGTCACGGACCTCAACTGAGGGCGGGCCATCTCCCAAAGCGGA...,NT96
1,CGCACACGCCGGGCCATGTGCCATACGACCCGCACGGCCCTTCGAG...,NT96
2,CCGAGATGTAGGATATGTTTGGTATCCTGATAATATAGTCTGGACA...,NT96
3,GACGTGACCCGGGGAGATGTGTTGCGGTCCGCTTGGGTCATTGACA...,NT96
4,CCTCCGTACTCGCCCGGGATTCGACTAACCTCCACTTTCGAACTCT...,NT96


### 3d. Combine all sources into one sequence table

In [10]:
# Stitch every source together. We don't keep the original per-source IDs (SeqID vs transcript_id
# are incompatible schemes) - a uniform index is simpler for everything that follows.
seqs = pd.concat([
    jura_dataset[[SEQ_COL, GROUP_COL]],
    human_random_df[[SEQ_COL, GROUP_COL]],
    human_truncated_df[[SEQ_COL, GROUP_COL]],
], ignore_index=True)
seqs.index = [f"seq_{i}" for i in range(len(seqs))]
seqs.index.name = "SeqID"
seqs = seqs[seqs['Sequence'].str.len() >= MIN_SEQ_LEN]
missing_groups = [g for g in GROUPS if g not in seqs[GROUP_COL].unique()]
if missing_groups:
    print("WARNING - groups not found in data:", missing_groups)

seqs.groupby(GROUP_COL).size().reindex(GROUPS)


Group
NT96                  10000
NT192                 10000
NT384                 10000
Human                 36830
human_truncated       67935
Random_gc50           10000
Random_gc_adjusted    10000
dtype: int64

In [11]:
# Split into one dataframe per group for convenience in every marker section below.
seqs_by_group = {g: seqs.loc[seqs[GROUP_COL] == g].copy() for g in GROUPS}
for g in GROUPS:
    print(f"{g:>16s}: {len(seqs_by_group[g])} sequences")


            NT96: 10000 sequences
           NT192: 10000 sequences
           NT384: 10000 sequences
           Human: 36830 sequences
 human_truncated: 67935 sequences
     Random_gc50: 10000 sequences
Random_gc_adjusted: 10000 sequences


### 3e. Extract the 40-dim embedding from the trained CNN

Load the trained ribosome-loading CNN (`models/params/JSD_trained_h100.pt`, architecture
in `models/optimus5prime.py`) and run every sequence through it using the model-agnostic
inference helpers in `models/helper.py`, adding the model's 40-dim pre-output embedding
(the ReLU'd `fc1` activation, right before dropout/the final regression layer) as a new
`embedding` column on each group's dataframe.

The model was trained on 50 nt UTRs immediately upstream of the start codon, so each
sequence is truncated to its last 50 nt before being one-hot encoded — safe here since
every remaining sequence is already `>= MIN_SEQ_LEN` (50 nt).

In [12]:
MODEL_PARAMS_PATH = "models/params/JSD_trained_h100.pt"

sys.path.append("models")
import torch
import torch.nn.functional as F
import helper
from optimus5prime import Model, INPUT_LENGTH

mrl_model = helper.load_model(Model(), MODEL_PARAMS_PATH)

def embed(model, seqs, **kwargs):
    encoded = helper.one_hot(seqs, length=INPUT_LENGTH)
    return helper.activations(model, encoded, model.fc1, transform=F.relu, **kwargs)

for g in GROUPS:
    seqs_by_group[g]["embedding"] = list(embed(mrl_model, seqs_by_group[g][SEQ_COL]))

seqs_by_group[GROUPS[0]][[SEQ_COL, "embedding"]].head()

/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/torch/nn/modules/conv.py:370: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Convolution.cpp:1037.)
  return F.conv1d(


,Sequence,embedding
SeqID,,
seq_0,GGATGTCACGGACCTCAACTGAGGGCGGGCCATCTCCCAAAGCGGA...,"[0.0, 2.044373, 1.1536115, 1.0966386, 0.0, 0.0..."
seq_1,CGCACACGCCGGGCCATGTGCCATACGACCCGCACGGCCCTTCGAG...,"[0.0, 0.06408875, 0.9421935, 0.58656245, 0.0, ..."
seq_2,CCGAGATGTAGGATATGTTTGGTATCCTGATAATATAGTCTGGACA...,"[0.0, 0.0, 1.2050023, 1.0272435, 0.0, 0.0, 0.0..."
seq_3,GACGTGACCCGGGGAGATGTGTTGCGGTCCGCTTGGGTCATTGACA...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
seq_4,CCTCCGTACTCGCCCGGGATTCGACTAACCTCCACTTTCGAACTCT...,"[0.0, 0.5047225, 3.2079756, 2.298248, 0.0, 1.3..."


### 3f. Predict MRL for every sequence

The network was trained to predict a z-scored MRL, so its raw output (from `model.out`,
the same tensor `embed` above taps at `model.fc1`, just one layer further) has to be run
through the inverse of that z-scoring to get back real MRL units. `helper.fit_scaler`
refits that scaler on the same two training files (`JSD_random_seqs.csv` /
`human_train.csv`) the model was originally trained against.

In [13]:
mrl_scaler = helper.fit_scaler(["JSD_random_seqs.csv", "human_train.csv"], column="rl", data_dir="data")

def predict(model, seqs, scaler, **kwargs):
    encoded = helper.one_hot(seqs, length=INPUT_LENGTH)
    scaled = helper.activations(model, encoded, model.out, **kwargs)
    return scaler.inverse_transform(scaled).reshape(-1)

for g in GROUPS:
    seqs_by_group[g]["MRL"] = predict(mrl_model, seqs_by_group[g][SEQ_COL], mrl_scaler)

seqs_by_group[GROUPS[0]][[SEQ_COL, "MRL"]].head()

,Sequence,MRL
SeqID,,
seq_0,GGATGTCACGGACCTCAACTGAGGGCGGGCCATCTCCCAAAGCGGA...,6.616736
seq_1,CGCACACGCCGGGCCATGTGCCATACGACCCGCACGGCCCTTCGAG...,5.376739
seq_2,CCGAGATGTAGGATATGTTTGGTATCCTGATAATATAGTCTGGACA...,5.547073
seq_3,GACGTGACCCGGGGAGATGTGTTGCGGTCCGCTTGGGTCATTGACA...,3.113982
seq_4,CCTCCGTACTCGCCCGGGATTCGACTAACCTCCACTTTCGAACTCT...,7.658539


### 3g. Sanity check: replay the embedding through the output layer

`embed` taps the network right after `fc1` + ReLU; `predict` above runs the *same* forward
pass one layer further, through `drop1` and `out`. In `model.eval()` mode `drop1` is a
no-op, so re-running the already-computed `embedding` column through `mrl_model.drop1`
and `mrl_model.out` by hand should reproduce `predict`'s raw (pre-inverse-transform)
output almost exactly. If the checkpoint's `out`/`fc1` weights didn't line up (wrong file,
architecture mismatch, a hook capturing the wrong layer, ...) this would show up as a poor
correlation here even though both `embed` and `predict` "ran" without error.

In [14]:
from scipy.stats import pearsonr

device = next(mrl_model.parameters()).device
check_embeddings = np.concatenate([np.stack(seqs_by_group[g]["embedding"].values) for g in GROUPS])
check_mrl = np.concatenate([seqs_by_group[g]["MRL"].values for g in GROUPS])

with torch.no_grad():
    replayed_scaled = mrl_model.out(mrl_model.drop1(torch.from_numpy(check_embeddings).to(device))).cpu().numpy()
replayed_mrl = mrl_scaler.inverse_transform(replayed_scaled).reshape(-1)

r, p = pearsonr(replayed_mrl, check_mrl)
print(f"correlation between replayed-embedding MRL and predict()'s MRL: r={r:.6f} (p={p:.3g})")
print(f"max abs difference: {np.max(np.abs(replayed_mrl - check_mrl)):.3e}")

correlation between replayed-embedding MRL and predict()'s MRL: r=1.000000 (p=0)
max abs difference: 0.000e+00


### 3i. Interactive 2D and 3D UMAP in a separate window

Same 40-dim embedding as above, but opened in the browser (`renderer="browser"`) so the
plots can be rotated/zoomed full-screen. Groups use the colorblind-safe Okabe-Ito palette
(yellow dropped for contrast on white), which separates the 7 groups much more clearly than
the pastel `Set2` palette. Click a group in the legend to hide it, double-click to isolate it.

In [18]:
# Okabe-Ito palette (colorblind-safe, high contrast) - one clearly distinct color per group
UMAP_PALETTE = dict(zip(GROUPS, [
    "#000000",  # black
    "#E69F00",  # orange
    "#56B4E9",  # sky blue
    "#009E73",  # bluish green
    "#0072B2",  # blue
    "#D55E00",  # vermillion
    "#CC79A7",  # reddish purple
]))
TEAL_CORAL_VIOLET = dict(zip(GROUPS, [
    "#7EF0D8",  # pale aqua
    "#2DD4BF",  # teal
    "#14A396",  # deep teal
    "#FF6B5B",  # coral
    "#C4A8FF",  # lavender
    "#8B5CF6",  # violet
]))

# 2D: reuse the 2D UMAP fitted above (same embedding, same parameters)
fig_2d = px.scatter(umap_df, x="UMAP 1", y="UMAP 2", color=GROUP_COL,
                    category_orders={GROUP_COL: GROUPS}, color_discrete_map=TEAL_CORAL_VIOLET,
                    opacity=0.6, render_mode="webgl",
                    title="2D UMAP of the 40-dim CNN embedding")
fig_2d.update_traces(marker=dict(size=3))
fig_2d.update_layout(legend=dict(itemsizing="constant"))
fig_2d.show(renderer="browser")   # opens in a separate browser window/tab

In [20]:
# 3D: needs its own UMAP fit with 3 output components
reducer_3d = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=3, random_state=RANDOM_SEED)
umap_coords_3d = reducer_3d.fit_transform(all_embeddings)

umap3d_df = pd.DataFrame(umap_coords_3d, columns=["UMAP 1", "UMAP 2", "UMAP 3"])
umap3d_df[GROUP_COL] = all_embedding_groups

fig_3d = px.scatter_3d(umap3d_df, x="UMAP 1", y="UMAP 2", z="UMAP 3", color=GROUP_COL,
                       category_orders={GROUP_COL: GROUPS}, color_discrete_map=TEAL_CORAL_VIOLET,
                       opacity=0.6, title="3D UMAP of the 40-dim CNN embedding")
fig_3d.update_traces(marker=dict(size=1.5))
fig_3d.update_layout(legend=dict(itemsizing="constant"))
fig_3d.show(renderer="browser")   # opens in a separate browser window/tab

/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
